# NusaSR: benchmark model TFLite

Membandingkan model super-resolution `.tflite` untuk app **tanpa training**. Kerusakan gambar sama dengan `NusaSR_FineTune_Lapangan_Colab.ipynb`: kondisi lapangan (gelap, kabut, goyang, silau, warna), diperkecil ke 160/256/384 px, lalu WebP 2/4/8 KB seperti `LoraImageCodec`.

Model dijalankan **persis seperti app** (`SrTensors` dan `SrTiling`): input `uint8` 128×128, tepi diisi piksel terdekat, gambar besar dipotong 128 px dengan tumpang-tindih 16 px.

**Cara pakai**

1. Upload semua file `.tflite` yang mau dibandingkan ke Drive folder `NusaSR_Bench/models`. Nama file dipakai sebagai nama metode. Model yang namanya mengandung `lapangan` dianggap model app sekarang (pembanding).
2. Opsional tapi paling penting: foto lapangan asli di Drive `NusaSR/foto_lapangan`, folder yang sama dengan notebook training. Foto ini hanya diuji, tidak dilatih.
3. Runtime CPU cukup, karena TFLite di sini tidak memakai GPU. Lalu **Run all**.

Hasil tersimpan di `NusaSR_Bench`: `hasil_benchmark.txt`, `per_gambar.csv`, dan `contoh.jpg`. Kecepatan di sini adalah CPU Colab, **bukan HP**. Ukur kecepatan HP lewat menu Uji AI di app setelah model dipasang.

In [ ]:
# COLAB_ONLY
!pip install -q lpips scikit-image
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# CONFIG
import csv, glob, hashlib, io, math, os, random, time, urllib.request, zipfile
from multiprocessing.pool import ThreadPool
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageOps

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter
try:
    import torch  # hanya untuk LPIPS
except ImportError:
    torch = None

WORK = Path("/content/nusasr_bench")
BENCH = Path("/content/drive/MyDrive/NusaSR_Bench")
MODELS_DIR = BENCH / "models"
FIELD_HR = Path("/content/drive/MyDrive/NusaSR/foto_lapangan")  # sama dengan notebook training
KODAK_HR = WORK / "kodak"
VAL_HR = WORK / "DIV2K_valid_HR"
REFERENCE = "lapangan"   # bagian nama file model app sekarang

PRESETS = [(160, 2000), (256, 4000), (384, 8000)]  # (sisi terpanjang px, batas byte), sama dengan LoraImageQuality
CONDITIONS = ("gelap", "kabut", "goyang", "silau", "warna")
VARIANTS = 2          # DIV2K: 1 asli + 1 kondisi acak, sama dengan validasi training
KODAK_LIMIT = 12      # 0 = semua 24 foto; tiap foto diuji di 6 kondisi x 3 preset
DIV2K_LIMIT = 0       # >0 untuk ikut menguji DIV2K valid (unduhan ±450 MB)
FIELD_LIMIT = 0       # 0 = semua foto lapangan
SCALE, TILE, OVERLAP = 4, 128, 16   # sama dengan SuperResolution.SIZE dan SrTiling.OVERLAP
THREADS = os.cpu_count() or 2
SEED = 42
SAMPLES = ("kodim05", "kodim20", "kodim23")

for d in (WORK, BENCH, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
print("thread TFLite:", THREADS, "| LPIPS di:", DEVICE)

In [ ]:
# COLAB_ONLY
def fetch(url, dest):
    """Unduh ke file .part dulu, jadi unduhan yang terputus tidak dianggap selesai."""
    dest = Path(dest)
    if dest.exists():
        return
    part = dest.with_name(dest.name + ".part")
    print("unduh", url, flush=True)
    urllib.request.urlretrieve(url, part)
    part.rename(dest)


KODAK_HR.mkdir(parents=True, exist_ok=True)
for i in range(1, 25):
    fetch(f"https://r0k.us/graphics/kodak/kodak/kodim{i:02d}.png", KODAK_HR / f"kodim{i:02d}.png")
if DIV2K_LIMIT and not VAL_HR.exists():
    z = WORK / "DIV2K_valid_HR.zip"
    fetch("http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip", z)
    with zipfile.ZipFile(z) as f:
        f.extractall(WORK)
    z.unlink()
print("data siap")

## Kerusakan seperti di app

Disalin dari notebook training supaya foto uji rusak dengan cara dan angka acak yang sama.

In [ ]:
def seed_of(text):
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16)


def apply_condition(img, kind, seed):
    if kind == "asli":
        return img
    rng = np.random.default_rng(seed)
    a = np.asarray(img, dtype=np.float32) / 255.0
    if kind == "gelap":  # malam atau dalam ruangan tanpa lampu, dengan noise sensor
        a = a ** rng.uniform(1.2, 2.0) * rng.uniform(0.25, 0.6)
        read, shot = rng.uniform(0.004, 0.015), rng.uniform(0.01, 0.03)
        a = a + rng.standard_normal(a.shape, dtype=np.float32) * (read + shot * np.sqrt(a))
    elif kind == "kabut":  # asap, debu, hujan rapat
        t = rng.uniform(0.45, 0.8)
        air = rng.uniform(0.7, 0.95) + rng.uniform(-0.03, 0.03, 3)
        a = a * t + air * (1 - t)
    elif kind == "goyang":  # tangan goyang saat memotret
        length = max(3, int(max(a.shape[:2]) * rng.uniform(0.004, 0.012)))
        angle = rng.uniform(0, math.pi)
        acc = np.zeros_like(a)
        for i in range(length):
            d = i - (length - 1) / 2
            acc += np.roll(a, (int(round(d * math.sin(angle))), int(round(d * math.cos(angle)))), axis=(0, 1))
        a = acc / length
    elif kind == "silau":  # terlalu terang atau melawan cahaya
        a = 1 - (1 - a) ** rng.uniform(1.6, 2.6)
    elif kind == "warna":  # white balance kamera melenceng
        gain = rng.uniform(0.8, 1.2, 3)
        a = a * (gain / gain.mean())
    out = Image.fromarray((np.clip(a, 0, 1) * 255).round().astype(np.uint8))
    buf = io.BytesIO()
    out.save(buf, "JPEG", quality=int(rng.integers(75, 96)))  # disimpan seperti kamera HP
    return Image.open(io.BytesIO(buf.getvalue())).convert("RGB")


def random_kinds(stem):
    kinds = ["asli"]
    for v in range(1, VARIANTS):
        kinds.append(CONDITIONS[seed_of(f"{stem}-{v}") % len(CONDITIONS)])
    return kinds


def all_kinds(stem):
    return ["asli", *CONDITIONS]


def only_original(stem):
    return ["asli"]


QUALITY_MIN, QUALITY_MAX, MIN_SIDE, SIDE_SHRINK = 5, 90, 64, 0.85


def android_downscale(src, target):
    long_side = max(src.size)
    sample = 1
    while long_side // (sample * 2) >= target * 2:
        sample *= 2
    img = src.reduce(sample) if sample > 1 else src
    s = target / max(img.size)
    return img.resize((max(1, int(img.width * s + 0.5)), max(1, int(img.height * s + 0.5))), Image.BILINEAR)


def webp_bytes(img, quality):
    buf = io.BytesIO()
    img.save(buf, "WEBP", quality=quality)
    return buf.getvalue()


def lora_degrade(src, max_side, max_bytes):
    side = max_side
    while True:
        small = android_downscale(src, side)
        lo, hi, best = QUALITY_MIN, QUALITY_MAX, None
        while lo <= hi:
            mid = (lo + hi) // 2
            data = webp_bytes(small, mid)
            if len(data) <= max_bytes:
                best, lo = data, mid + 1
            else:
                hi = mid - 1
        if best is not None or side <= MIN_SIDE:
            data = best if best is not None else webp_bytes(small, QUALITY_MIN)
            return Image.open(io.BytesIO(data)).convert("RGB")
        side = max(MIN_SIDE, int(side * SIDE_SHRINK))


def gt_path(lr_path):
    return lr_path[:-len("lr.npy")] + "gt.npy"


def preset_of(lr_path):
    return int(Path(lr_path).stem.rsplit("_", 2)[1])


def kind_of(lr_path):
    return Path(lr_path).stem.rsplit("_", 2)[0].rsplit("-", 1)[1]


def photo_of(lr_path):
    return Path(lr_path).stem.rsplit("_", 2)[0].rsplit("-", 1)[0]


def make_pairs(args):
    """Satu foto + satu kondisi -> pasangan uji untuk ketiga preset."""
    path, out, kind = args
    stem = Path(path).stem.replace("_", "-").replace(" ", "-")
    names = [(out / f"{stem}-{kind}_{side}_lr.npy", side, size) for side, size in PRESETS]
    if all(p.exists() and Path(gt_path(str(p))).exists() for p, _, _ in names):
        return [str(p) for p, _, _ in names]
    try:
        src = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    except Exception as e:
        print("lewati", path, e)
        return []
    src = apply_condition(src, kind, seed_of(f"{stem}-{kind}"))
    made = []
    for lr_path, side, size in names:
        lr = lora_degrade(src, side, size)
        want = (lr.width * SCALE, lr.height * SCALE)
        # Foto lebih kecil dari 4x gambar kiriman dinilai pada ukuran aslinya.
        gt = src if want[0] > src.width or want[1] > src.height else src.resize(want, Image.LANCZOS)
        np.save(gt_path(str(lr_path)), np.asarray(gt, dtype=np.uint8))
        np.save(lr_path, np.asarray(lr, dtype=np.uint8))
        made.append(str(lr_path))
    return made


def image_files(folder):
    files = []
    for ext in ("png", "jpg", "jpeg", "webp"):
        for e in (ext, ext.upper()):
            files += glob.glob(f"{folder}/**/*.{e}", recursive=True)
    return sorted(set(files))


def build_pairs(files, out, kinds_of):
    out.mkdir(parents=True, exist_ok=True)
    jobs = [(p, out, k) for p in files for k in kinds_of(Path(p).stem.replace("_", "-"))]
    with ThreadPool(os.cpu_count() or 2) as pool:
        made = [x for group in pool.imap_unordered(make_pairs, jobs) for x in group]
    print(f"{out.name}: {len(files)} foto, {len(made)} pasangan", flush=True)
    return sorted(made)


kodak_files = sorted(image_files(KODAK_HR), key=lambda p: (Path(p).stem not in SAMPLES, p))[:KODAK_LIMIT or None]
kodak_pairs = build_pairs(kodak_files, WORK / "pairs_kodak", all_kinds)
val_files = image_files(VAL_HR)[:DIV2K_LIMIT] if DIV2K_LIMIT else []
val_pairs = build_pairs(val_files, WORK / "pairs_val", random_kinds) if val_files else []
field_files = image_files(FIELD_HR) if FIELD_HR.exists() else []
if FIELD_LIMIT:
    field_files = field_files[:FIELD_LIMIT]
field_pairs = build_pairs(field_files, WORK / "pairs_field", only_original) if field_files else []
if not field_pairs:
    print("Foto lapangan tidak ditemukan di", FIELD_HR, "- hasil belum mewakili kondisi nyata.")

## Metrik

PSNR dan SSIM pada kanal Y (makin tinggi makin baik), LPIPS AlexNet (makin rendah makin baik). Sama dengan notebook training.

In [ ]:
try:
    import lpips
    LPIPS_NET = lpips.LPIPS(net="alex", verbose=False).to(DEVICE).eval()
except Exception as e:
    LPIPS_NET = None
    print("LPIPS dilewati:", e)
try:
    from skimage.metrics import structural_similarity
except Exception as e:
    structural_similarity = None
    print("SSIM dilewati:", e)


def to_tensor(a):
    return torch.from_numpy(a.copy()).permute(2, 0, 1).float() / 255.0


def luma(a):
    return 16 + (65.481 * a[..., 0] + 128.553 * a[..., 1] + 24.966 * a[..., 2]) / 255.0


def psnr(a, b):
    return 10 * math.log10(255 ** 2 / max(np.mean((luma(a.astype(float)) - luma(b.astype(float))) ** 2), 1e-10))


def ssim(a, b):
    return structural_similarity(luma(a.astype(float)), luma(b.astype(float)), data_range=255) if structural_similarity else float("nan")


def lpips_of(a, b):
    if LPIPS_NET is None:
        return float("nan")
    t = lambda x: (to_tensor(x)[None].to(DEVICE) * 2 - 1)
    with torch.no_grad():
        return float(LPIPS_NET(t(a), t(b)))


def fit(out, gt):
    if out.shape != gt.shape:
        out = np.asarray(Image.fromarray(out).resize((gt.shape[1], gt.shape[0]), Image.LANCZOS))
    return out


def bicubic(lr):
    return np.asarray(Image.fromarray(lr).resize((lr.shape[1] * SCALE, lr.shape[0] * SCALE), Image.BICUBIC))

## Model dijalankan seperti app

`SrTensors.encodeUint8/decodeUint8` dan `SrTiling.plan/stitch`: pembulatan setengah ke atas seperti `roundToInt()`, koordinat di luar gambar mengulang piksel tepi, dan hanya bagian inti tiap potongan yang ditempel.

In [ ]:
def app_crop(img, x0, y0, size=TILE):
    h, w = img.shape[:2]
    ys = np.clip(np.arange(y0, y0 + size), 0, h - 1)
    xs = np.clip(np.arange(x0, x0 + size), 0, w - 1)
    return img[ys][:, xs]


def app_plan(w, h):
    step, tiles = TILE - 2 * OVERLAP, []
    for y in range(0, h, step):
        core_h = min(step, h - y)
        for x in range(0, w, step):
            tiles.append((x - OVERLAP, y - OVERLAP, x, y, min(step, w - x), core_h))
    return tiles


class AppModel:
    def __init__(self, path):
        self.path = Path(path)
        self.name = self.path.stem
        self.it = Interpreter(model_path=str(self.path), num_threads=THREADS)
        self.it.allocate_tensors()
        self.inp, self.out = self.it.get_input_details()[0], self.it.get_output_details()[0]
        shapes = (list(self.inp["shape"]), list(self.out["shape"]))
        if shapes != ([1, TILE, TILE, 3], [1, TILE * SCALE, TILE * SCALE, 3]) or \
                self.inp["dtype"] != np.uint8 or self.out["dtype"] != np.uint8:
            raise ValueError(f"bentuk/tipe beda dari app: {shapes}, {self.inp['dtype']} -> {self.out['dtype']}")
        self.in_scale, self.in_zero = self.inp["quantization"]
        self.out_scale, self.out_zero = self.out["quantization"]
        self.probe_ms = float("nan")

    def tile(self, a):
        v = a.astype(np.float32) / np.float32(255)
        q = np.clip(np.floor(v / np.float32(self.in_scale) + self.in_zero + 0.5), 0, 255).astype(np.uint8)
        self.it.set_tensor(self.inp["index"], q[None])
        self.it.invoke()
        raw = self.it.get_tensor(self.out["index"])[0].astype(np.float32)
        o = (raw - self.out_zero) * np.float32(self.out_scale) * np.float32(255)
        return np.clip(np.floor(o + 0.5), 0, 255).astype(np.uint8)

    def __call__(self, lr):
        h, w = lr.shape[:2]
        if max(h, w) <= TILE:  # SuperResolution.upscale: sekali jalan, gambar di kiri atas
            return self.tile(app_crop(lr, 0, 0))[:h * SCALE, :w * SCALE]
        out = np.zeros((h * SCALE, w * SCALE, 3), np.uint8)
        for sx, sy, cx, cy, cw, ch in app_plan(w, h):
            up = self.tile(app_crop(lr, sx, sy))
            ox, oy = (cx - sx) * SCALE, (cy - sy) * SCALE
            out[cy * SCALE:(cy + ch) * SCALE, cx * SCALE:(cx + cw) * SCALE] = up[oy:oy + ch * SCALE, ox:ox + cw * SCALE]
        return out


paths = sorted(glob.glob(str(MODELS_DIR / "*.tflite")))
assert paths, f"Belum ada model. Upload file .tflite ke {MODELS_DIR}"
models = []
for p in paths:
    try:
        models.append(AppModel(p))
    except Exception as e:
        print("lewati", Path(p).name, ":", e)
models.sort(key=lambda m: (REFERENCE not in m.name, m.name))
ref = next((m for m in models if REFERENCE in m.name), None)
print("pembanding (model app sekarang):", ref.name if ref else "tidak ada, tabel selisih dilewati")

probe = np.random.default_rng(SEED).integers(0, 256, (TILE, TILE, 3), dtype=np.uint8)
for m in models:
    m.tile(probe)  # pemanasan
    times = []
    for _ in range(3):
        t = time.perf_counter(); m.tile(probe); times.append((time.perf_counter() - t) * 1000)
    m.probe_ms = sorted(times)[1]
    print(f"{m.name:<40}{m.probe_ms:>7.0f} ms per potongan 128 px  {m.path.stat().st_size / 1024:>6.0f} KB")

all_pairs = kodak_pairs + val_pairs + field_pairs
tiles = 0
for p in all_pairs:
    h, w = np.load(p, mmap_mode="r").shape[:2]
    tiles += 1 if max(h, w) <= TILE else len(app_plan(w, h))
print(f"\n{len(all_pairs)} pasangan, {tiles} potongan per model, perkiraan ±{sum(m.probe_ms for m in models) * tiles / 60000:.0f} menit."
      " Kecilkan KODAK_LIMIT kalau terlalu lama.")

## Hasil

Tiap metode dijalankan sekali per pasangan, lalu dirata-rata per kondisi dan per preset.

In [ ]:
methods = [("bicubic", bicubic)] + [(m.name, m) for m in models]
results = {}


def score_all(pairs, label):
    t0 = time.time()
    for name, fn in methods:
        for p in pairs:
            if (name, p) in results:
                continue
            lr, gt = np.load(p), np.load(gt_path(p))
            out = fit(fn(lr), gt)
            results[(name, p)] = (psnr(gt, out), ssim(gt, out), lpips_of(gt, out))
        print(f"  {label}: {name} selesai ({(time.time() - t0) / 60:.1f} menit)", flush=True)


for label, pairs in (("Kodak", kodak_pairs), ("DIV2K valid", val_pairs), ("Foto lapangan", field_pairs)):
    if pairs:
        score_all(pairs, label)

lines = []


def table(title, pairs, key, key_name):
    start = len(lines)
    lines.append(f"\n== {title} ({len(pairs)} pasangan) ==")
    lines.append(f"{'metode':<40}{key_name:>8}{'PSNR':>9}{'SSIM':>9}{'LPIPS':>9}")
    for name, _ in methods:
        groups = {}
        for p in pairs:
            groups.setdefault(key(p), []).append(results[(name, p)])
        for k, rows in sorted(groups.items()):
            a, b, c = np.mean(rows, axis=0)
            lines.append(f"{name:<40}{str(k):>8}{a:>9.3f}{b:>9.4f}{c:>9.4f}")
    print("\n".join(lines[start:]), flush=True)


table("Kodak per kondisi (rata-rata 3 preset)", kodak_pairs, kind_of, "kondisi")
table("Kodak per preset (semua kondisi)", kodak_pairs, preset_of, "preset")
if val_pairs:
    table("DIV2K valid per preset", val_pairs, preset_of, "preset")
if field_pairs:
    table("Foto lapangan per preset", field_pairs, preset_of, "preset")
else:
    lines.append("\nFoto lapangan belum ada: hasil belum mewakili kondisi nyata.")


def mean_of(name, pairs):
    return np.mean([results[(name, p)] for p in pairs], axis=0)


start = len(lines)
if ref:
    focus = [("foto lapangan", field_pairs),
             ("Kodak asli", [p for p in kodak_pairs if kind_of(p) == "asli"]),
             ("Kodak kondisi lapangan", [p for p in kodak_pairs if kind_of(p) != "asli"]),
             ("DIV2K valid", val_pairs)]
    lines.append(f"\n== Selisih terhadap model app sekarang ({ref.name}) ==")
    lines.append("PSNR dan SSIM: + berarti lebih baik. LPIPS: - berarti lebih baik.")
    lines.append(f"{'metode':<40}{'data':<24}{'PSNR':>9}{'SSIM':>9}{'LPIPS':>9}")
    for name, _ in methods:
        if name == ref.name:
            continue
        for label, pairs in focus:
            if pairs:
                d = mean_of(name, pairs) - mean_of(ref.name, pairs)
                lines.append(f"{name:<40}{label:<24}{d[0]:>+9.3f}{d[1]:>+9.4f}{d[2]:>+9.4f}")
lines.append(f"\n== Kecepatan TFLite (CPU Colab {THREADS} thread, bukan HP) ==")
for m in models:
    lines.append(f"{m.name:<40}{m.probe_ms:>7.0f} ms/potongan  {m.path.stat().st_size / 1024:>6.0f} KB")
print("\n".join(lines[start:]))

(BENCH / "hasil_benchmark.txt").write_text("\n".join(lines), encoding="utf-8")
with open(BENCH / "per_gambar.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["metode", "dataset", "foto", "kondisi", "preset", "psnr", "ssim", "lpips"])
    for (name, p), (a, b, c) in sorted(results.items()):
        w.writerow([name, Path(p).parent.name, photo_of(p), kind_of(p), preset_of(p), f"{a:.4f}", f"{b:.5f}", f"{c:.5f}"])
print("\nTersimpan:", BENCH / "hasil_benchmark.txt", "dan", BENCH / "per_gambar.csv")

In [ ]:
def sample_sheet(path, preset=256, kinds=("asli", "gelap", "kabut", "goyang")):
    """Gambar contoh: satu baris per foto+kondisi, kolom = metode lalu foto asli."""
    rows = [p for p in kodak_pairs if preset_of(p) == preset and kind_of(p) in kinds
            and any(Path(p).name.startswith(s + "-") for s in SAMPLES)]
    if not rows:
        return
    cell_w = 360
    names = [n for n, _ in methods] + ["foto asli"]
    first = np.load(gt_path(rows[0]))
    cell_h = int(cell_w * first.shape[0] / first.shape[1])
    sheet = Image.new("RGB", (cell_w * len(names), (cell_h + 20) * len(rows) + 20), "white")
    draw = ImageDraw.Draw(sheet)
    for j, n in enumerate(names):
        draw.text((j * cell_w + 4, 4), n, fill="black")
    for i, p in enumerate(rows):
        lr, gt = np.load(p), np.load(gt_path(p))
        outs = [fit(fn(lr), gt) for _, fn in methods] + [gt]
        y = 20 + i * (cell_h + 20)
        draw.text((4, y + 2), f"{Path(p).stem.rsplit('_', 2)[0]}  {preset}px", fill="black")
        for j, o in enumerate(outs):
            im = Image.fromarray(o)
            box = (im.width // 4, im.height // 4, im.width * 3 // 4, im.height * 3 // 4)  # potongan tengah, diperbesar
            sheet.paste(im.crop(box).resize((cell_w, cell_h), Image.LANCZOS), (j * cell_w, y + 18))
    sheet.save(path, quality=90)
    print("contoh:", path)


sample_sheet(BENCH / "contoh.jpg")

## Cara membaca

- Lihat dulu tabel **selisih terhadap model app sekarang**. Model baru layak menggantikan kalau di **foto lapangan** PSNR dan SSIM naik serta LPIPS turun, dan tidak kalah jauh di **Kodak asli**.
- Jangan pilih model yang hanya menang di kondisi buatan tetapi kalah di foto lapangan.
- PSNR naik tapi LPIPS memburuk (atau sebaliknya) berarti model lebih "halus" atau lebih "mengarang detail". Cek `contoh.jpg` sebelum memutuskan.
- Kecepatan di sini CPU Colab. App memilih model pertama yang ≤ 1.500 ms per potongan di HP (`SuperResolution.MAX_MS`), jadi ukur ulang di HP lewat menu Uji AI.